In [1]:
from dotenv import load_dotenv
from langchain_core.tools import tool


load_dotenv()


True

Google Gemini

In [2]:
from langchain.chat_models import init_chat_model

# Initialize the Google Gemini model using init_chat_model with provider prefix
llm = init_chat_model("google_genai:gemini-3.5-flash")

Groq

In [3]:
# Initialize the Google Gemini model using init_chat_model with provider prefix
llm = init_chat_model("groq:llama-3.1-8b-instant")

In [4]:
# Test invoking the model
response = llm.invoke("Give me a essay about a Pet in 1000 words")
print(response.content)

**The Joy and Responsibility of Having a Pet**

Having a pet is a life-changing experience that brings immense joy, companionship, and love into one's life. Pets, whether they be dogs, cats, birds, or even fish, have the unique ability to provide unconditional love and acceptance, making them an integral part of many families around the world. However, having a pet is not just a matter of adopting a cute and cuddly creature; it is a significant responsibility that requires a tremendous amount of time, effort, and commitment.

One of the primary benefits of having a pet is the companionship they provide. Pets are social animals that thrive on interaction and attention from their owners. Whether it's a quick walk around the block or a long, leisurely stroll, pets love to be by their owner's side, providing a sense of security and comfort. This companionship can be especially beneficial for people who live alone or are isolated from friends and family. Pets can help alleviate feelings of 

In [5]:
# Streaming
line = 1
for chunk in llm.stream("Give me a essay about a Pet in 100 words"):
    print('line', line)
    print(chunk.content)
    print()
    line += 1

line 1


line 2
A

line 3
 pet

line 4
 is

line 5
 a

line 6
 cherished

line 7
 companion

line 8
 that

line 9
 brings

line 10
 joy

line 11
 and

line 12
 companions

line 13
hip

line 14
 to

line 15
 our

line 16
 lives

line 17
.

line 18
 Whether

line 19
 it

line 20
's

line 21
 a

line 22
 furry

line 23
 cat

line 24
 or

line 25
 a

line 26
 wag

line 27
ging

line 28
 dog

line 29
,

line 30
 a

line 31
 pet

line 32
 is

line 33
 a

line 34
 loyal

line 35
 friend

line 36
 that

line 37
 provides

line 38
 unw

line 39
av

line 40
ering

line 41
 love

line 42
 and

line 43
 affection

line 44
.

line 45
 Having

line 46
 a

line 47
 pet

line 48
 can

line 49
 also

line 50
 have

line 51
 numerous

line 52
 physical

line 53
 and

line 54
 mental

line 55
 health

line 56
 benefits

line 57
,

line 58
 such

line 59
 as

line 60
 reduced

line 61
 stress

line 62
 and

line 63
 anxiety

line 64
.

line 65
 Pets

line 66
 encourage

line 67
 us

line 68
 to

line 69
 

In [6]:
# Batch

responses = llm.batch(["who are you?", "whats your capability", "what is AI?"], config={"max_concurrency" : 2})
responses

[AIMessage(content="I'm an artificial intelligence model known as a large language model (LLM) or a chatbot. I was created to assist and communicate with users like you through text-based conversations. I don't have a personal identity or emotions, but I'm designed to provide information, answer questions, and engage in discussions on a wide range of topics.\n\nI'm constantly learning and improving from the vast amount of text data I was trained on, which allows me to generate human-like responses to your queries. However, I'm not perfect and can make mistakes, so please feel free to point out any errors or inaccuracies you may encounter.\n\nI'm here to help, and I'll do my best to provide you with accurate and helpful information or simply chat with you about your interests. What would you like to talk about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 39, 'total_tokens': 203, 'completion_time': 0.202545748, 'completion_tokens_

In [8]:
# Tools calling
import json

import urllib

# ── Tool 1: Weather ──────────────────────────────────────────────────────────
@tool
def get_weather(location: str) -> str:
    """Get the current weather details for a given location.

    Args:
        location: The name of the city or location (e.g., 'London', 'New York').
    """
    try:
        safe_location = urllib.parse.quote(location)
        url = f"https://wttr.in/{safe_location}?format=3"
        req = urllib.request.Request(url, headers={"User-Agent": "curl/7.64.1"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            raw = resp.read().decode("utf-8")
            return raw.encode("ascii", "ignore").decode("ascii").strip()
    except Exception as e:
        return f"Error fetching weather for '{location}': {e}"


# ── Tool 2: Location Details ─────────────────────────────────────────────────
@tool
def get_location_details(city: str) -> str:
    """Get geocoding and location details (latitude, longitude, country, address) for a city.

    Args:
        city: The name of the city (e.g., 'Paris', 'Tokyo', 'San Francisco').
    """
    try:
        safe_city = urllib.parse.quote(city)
        url = (
            f"https://nominatim.openstreetmap.org/search"
            f"?q={safe_city}&format=json&addressdetails=1&limit=1"
        )
        # Nominatim requires a custom User-Agent
        req = urllib.request.Request(url, headers={"User-Agent": "LangChainLocationTool/1.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode("utf-8"))

        if not data:
            return f"No location details found for '{city}'."

        result  = data[0]
        address = result.get("address", {})
        return (
            f"Location Details for '{city}':\n"
            f"  Full address : {result.get('display_name')}\n"
            f"  Latitude     : {result.get('lat')}\n"
            f"  Longitude    : {result.get('lon')}\n"
            f"  State/Region : {address.get('state', 'N/A')}\n"
            f"  Country      : {address.get('country', 'N/A')}"
        )
    except Exception as e:
        return f"Error fetching location details for '{city}': {e}"




In [9]:
# ── Bind both tools to the LLM ───────────────────────────────────────────────
tools = [get_weather, get_location_details]
llm_with_tools = llm.bind_tools(tools)

# ── Send a query that requires both tools ────────────────────────────────────
query = "What is the weather in Paris right now, and where exactly is Paris located?"
ai_msg = llm_with_tools.invoke(query)

print("AI message tool_calls:", ai_msg.tool_calls)

# ── Execute each tool call returned by the model ─────────────────────────────
tool_map = {t.name: t for t in tools}

for tc in ai_msg.tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    print(f"\nCalling tool: {tool_name}  args: {tool_args}")
    result = tool_map[tool_name].invoke(tool_args)
    print("Result:")
    print(result)

AI message tool_calls: [{'name': 'get_weather', 'args': {'location': 'Paris'}, 'id': '6n32rjfdy', 'type': 'tool_call'}, {'name': 'get_location_details', 'args': {'city': 'Paris'}, 'id': 'xqas1hykm', 'type': 'tool_call'}]

Calling tool: get_weather  args: {'location': 'Paris'}
Result:
Paris:   +32C

Calling tool: get_location_details  args: {'city': 'Paris'}
Result:
Location Details for 'Paris':
  Full address : Paris, Île-de-France, France métropolitaine, France
  Latitude     : 48.8588897
  Longitude    : 2.3200410
  State/Region : Île-de-France
  Country      : France
